In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
# dbutils.notebook.run("/Workspace/Users/murariazure@gmail.com/demo_class_notes/silver/latest/common_functions", 0)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Change only these if your Unity Catalog names are different
CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

In [0]:
pit_stops=spark.table(bronze_table("pit_stops"))
pit_stops.printSchema()
print("Bronze rows:",pit_stops.count())

## 1. NULL + duplicate checks

In [0]:
display(pit_stops.filter(
    F.col("race_id").isNull() |
    F.col("driver_id").isNull() |
    F.col("stop").isNull()
))
display(pit_stops.groupBy("race_id","driver_id","stop")
                  .count().filter(F.col("count")>1))

## 2. String operations

In [0]:
pit_work=(
    pit_stops
    .withColumn("stop_clean",F.trim("stop"))
    .withColumn("time_clean",F.trim("time"))
    .withColumn("duration_clean",F.trim("duration"))
    .withColumn("duration_parts",F.split("duration_clean",":"))
    .withColumn("duration_length",F.length("duration_clean"))
    .withColumn("duration_prefix",F.substring("duration_clean",1,3))
    .withColumn("milliseconds_clean",F.col("milliseconds").cast("long"))
)
display(pit_work.select(
    "race_id","driver_id","stop","stop_clean",
    "time_clean","duration_clean","duration_parts",
    "duration_length","duration_prefix","milliseconds_clean"
).limit(20))

## 3. contains / startsWith / endsWith

In [0]:
display(pit_work.filter(F.col("duration_clean").contains(":"))
              .select("race_id","driver_id","duration_clean").limit(20))
display(pit_work.filter(F.col("stop_clean").startswith("1"))
              .select("race_id","driver_id","stop_clean").limit(20))
display(pit_work.filter(F.col("stop_clean").endswith("1"))
              .select("race_id","driver_id","stop_clean").limit(20))

## 4. Incremental batch

In [0]:
target_name=silver_table("pit_stops")

if not spark.catalog.tableExists(target_name):
    pit_stops_batch=pit_stops
else:
    last_ts=spark.table(target_name).agg(
        F.max("ingestion_timestamp").alias("max_ts")
    ).first()["max_ts"]

    print("Last Silver ingestion_timestamp:",last_ts)

    pit_stops_batch=(
        pit_stops if last_ts is None
        else pit_stops.filter(F.col("ingestion_timestamp")>F.lit(last_ts))
    )

print("Rows selected:",pit_stops_batch.count())

## 5. Clean selected batch

In [0]:
pit_stops_clean=(
    pit_stops_batch
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("stop").isNotNull())
    .dropDuplicates(["race_id","driver_id","stop"])
    .withColumn("stop_clean",F.trim("stop"))
    .withColumn("time_clean",F.trim("time"))
    .withColumn("duration_clean",F.trim("duration"))
    .withColumn("milliseconds_clean",F.col("milliseconds").cast("long"))
    .withColumn(
        "stop_category",
        F.when(F.col("stop_clean")=="1","First Stop")
         .when(F.col("stop_clean")=="2","Second Stop")
         .otherwise("Later Stop")
    )
    .withColumn("silver_processed_timestamp",F.current_timestamp())
)
display(pit_stops_clean.limit(20))

## 6. MERGE

In [0]:
if not spark.catalog.tableExists(target_name):
    (pit_stops_clean.write.format("delta").mode("overwrite")
     .option("overwriteSchema","true").saveAsTable(target_name))
else:
    target=DeltaTable.forName(spark,target_name)
    (target.alias("t")
     .merge(
         pit_stops_clean.alias("s"),
         "t.race_id=s.race_id AND t.driver_id=s.driver_id AND t.stop=s.stop"
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

print("PIT_STOPS MERGE completed.")

In [0]:
silver=spark.table(target_name)
print("Silver rows:",silver.count())
display(silver.orderBy(F.desc("ingestion_timestamp")).limit(20))